In [2]:

from transformers import AutoTokenizer,AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import torch
from peft import PeftModel, LoraConfig, get_peft_model
import pandas as pd
from tqdm import tqdm
import re
import time


from evaluation_models import prepare_data_to_necessary_from_for_evaluation, compute_all_metrics_responses,compute_metrics_for_classifier_only,generate_responses_for_evaluate_second_step_hubrid_system,compute_metrics_for_hybrid_system,prepare_data_for_win_rate_evaluation,win_rate
from prepare_data_for_dpo import generate_batch
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
data=prepare_data_to_necessary_from_for_evaluation('final_test_data/test_data_final_all_models.csv')
data=data.rename(columns={'is_unsafe_prompt':'is_unsafe_prompt_real',
                          'response_part':'response_part_real',
                          'analysis_part':'analysis_part_real',

                          'attack_type':'attack_type_real',
                          'confidence':'confidence_real',
                          'recommendation':'recommendation_real'})

In [ ]:
compute_all_metrics_responses(data,path_data='final_test_data/responses_base_model.csv',return_confidence=False)

Наличие аналитической части:78.8%
Полное соотвесвтие формату 28.3%
Acccuracy: 0.85
F1: 0.874
Precision: 0.827
Recall: 0.926


100%|██████████| 32/32 [00:24<00:00,  1.30it/s]



Процент плохих ответов: 6.800%
Среднее качество ответов: 0.9048592345952057
ROUGE-1: 0.2378
ROUGE-2: 0.0610
ROUGE-L: 0.1554
BLEU: 0.0285


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/32 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/16 [00:00<?, ?it/s]

done in 17.48 seconds, 57.22 sentences/sec
BERTScore Precision: 0.5630
BERTScore Recall:    0.5409
BERTScore F1:        0.5477
Средняя длина ответа: 494.172


In [13]:
compute_all_metrics_responses(data,path_data='final_test_data/responses_sft_2.csv',model_path='dpo_response_classifier',return_confidence=False)

Наличие аналитической части:85.2%
Полное соответствие формату 84.9%
Acccuracy: 0.822
F1: 0.823
Precision: 0.921
Recall: 0.744


100%|██████████| 32/32 [14:05<00:00, 26.41s/it]



Процент плохих ответов: 5.400%
Среднее качество ответов: 0.9283919317470863
ROUGE-1: 0.2283
ROUGE-2: 0.0726
ROUGE-L: 0.1520
BLEU: 0.0294


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 323.58it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


100%|██████████| 31/31 [03:23<00:00,  6.57s/it]


computing greedy matching.


100%|██████████| 16/16 [00:02<00:00,  6.79it/s]


done in 205.94 seconds, 4.86 sentences/sec
BERTScore Precision: 0.5759
BERTScore Recall:    0.5395
BERTScore F1:        0.5493
Средняя длина ответа: 976.186


In [ ]:
compute_all_metrics_responses(data,path_data='final_test_data/responses_dpo_extended_model.csv',model_path='dpo_response_classifier')

Наличие аналитической части:89.9%
Полное соотвесвтие формату 89.3%
Acccuracy: 0.817
F1: 0.81
Precision: 0.943
Recall: 0.709


100%|██████████| 32/32 [00:30<00:00,  1.04it/s]



Процент плохих ответов: 4.600%
Среднее качество ответов: 0.9396075858767144
ROUGE-1: 0.2410
ROUGE-2: 0.0779
ROUGE-L: 0.1599
BLEU: 0.0329


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/31 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/16 [00:00<?, ?it/s]

done in 17.17 seconds, 58.25 sentences/sec
BERTScore Precision: 0.5861
BERTScore Recall:    0.5468
BERTScore F1:        0.5586
Средняя длина ответа: 850.294


In [ ]:
first_step_labels=compute_metrics_for_classifier_only(data,model_path='prompts classifier')

100%|██████████| 32/32 [00:08<00:00,  3.80it/s]


8.445842266082764

In [ ]:
HF_TOKEN=os.getenv('HF_TOKEN')
model_name = "Qwen/Qwen3-4B"
tokenizer_dpo=AutoTokenizer.from_pretrained(model_name,token=HF_TOKEN)
tokenizer_dpo.pad_token=tokenizer_dpo.eos_token

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [ ]:
bnb_config_base_model=BitsAndBytesConfig(

    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
base_model_for_dpo_extended=AutoModelForCausalLM.from_pretrained(model_name,
                                                device_map='auto',
                                                token=HF_TOKEN,
                                                quantization_config=bnb_config_base_model,
                                                dtype=torch.float16)
dpo_model_extended=PeftModel.from_pretrained(base_model_for_dpo_extended,'dpo_model_extended')

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
CONFIG={'do_sample':False, 'repetition_penalty':1.1}


In [ ]:
first_step_labels, low_confidence_data=generate_responses_for_evaluate_second_step_hubrid_system(first_step_labels,dpo_model_extended,tokenizer_dpo,CONFIG)

Наличие аналитической части:85.3%
Полное соотвесвтие формату 85.3%


100%|██████████| 5/5 [00:04<00:00,  1.13it/s]



Процент плохих ответов: 7.692%
Среднее качество ответов: 0.910916123597824
ROUGE-1: 0.1954
ROUGE-2: 0.0428
ROUGE-L: 0.1195
BLEU: 0.0227


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/5 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/3 [00:00<?, ?it/s]

done in 4.06 seconds, 35.25 sentences/sec
BERTScore Precision: 0.5350
BERTScore Recall:    0.5204
BERTScore F1:        0.5206
Средняя длина ответа: 989.4615384615385


In [ ]:
compute_metrics_for_hybrid_system(first_step_labels, low_confidence_data)

Acccuracy: 0.973
F1: 0.973
Precision: 0.982
Recall: 0.965


# Win Rate

In [4]:
all_results_confidence_chosen_response=prepare_data_for_win_rate_evaluation(data)

100%|██████████| 32/32 [05:17<00:00,  9.93s/it]



Процент плохих ответов: 6.800%
Среднее качество ответов: 0.9048591774222441


100%|██████████| 32/32 [06:26<00:00, 12.09s/it]



Процент плохих ответов: 5.400%
Среднее качество ответов: 0.9297743253926747


100%|██████████| 32/32 [06:26<00:00, 12.07s/it]



Процент плохих ответов: 4.900%
Среднее качество ответов: 0.9312105816784315


100%|██████████| 32/32 [06:22<00:00, 11.97s/it]


Процент плохих ответов: 4.600%
Среднее качество ответов: 0.9396075433930382


In [6]:
win_rate(all_results_confidence_chosen_response['confidence_chosen_class_base'],all_results_confidence_chosen_response['confidence_chosen_class_sft'])

np.float64(0.555)

In [5]:
all_results_confidence_chosen_response.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   confidence_chosen_class_base                     1000 non-null   float64
 1   confidence_chosen_class_sft                      1000 non-null   float64
 2   confidence_chosen_class_dpo_without_sft_answers  1000 non-null   float64
 3   confidence_chosen_class_dpo_extended             1000 non-null   float64
dtypes: float64(4)
memory usage: 31.4 KB


In [7]:
win_rate(all_results_confidence_chosen_response['confidence_chosen_class_base'],all_results_confidence_chosen_response['confidence_chosen_class_dpo_without_sft_answers'])

np.float64(0.551)

In [8]:
win_rate(all_results_confidence_chosen_response['confidence_chosen_class_base'],all_results_confidence_chosen_response['confidence_chosen_class_dpo_extended'])

np.float64(0.5975)

In [9]:
win_rate(all_results_confidence_chosen_response['confidence_chosen_class_sft'],all_results_confidence_chosen_response['confidence_chosen_class_dpo_without_sft_answers'])

np.float64(0.507)

In [10]:
win_rate(all_results_confidence_chosen_response['confidence_chosen_class_sft'],all_results_confidence_chosen_response['confidence_chosen_class_dpo_extended'])

np.float64(0.531)

In [11]:
win_rate(all_results_confidence_chosen_response['confidence_chosen_class_dpo_without_sft_answers'],all_results_confidence_chosen_response['confidence_chosen_class_dpo_extended'])

np.float64(0.531)